Disclosure of AI Usage: The creation of this code was assisted by generative AI (Claude 4.6 Sonnet). We release all data and code for reproducibility and transparency.

---
## 1. Installation

In [ ]:
!pip install -q \
    transformers datasets \
    opencc-python-reimplemented \
    umap-learn scikit-learn \
    matplotlib seaborn pandas tqdm \
    sentencepiece protobuf \
    turkicnlp cyrtranslit pykakasi nltk aksharamukha

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
print('Installation complete.')

---
## 2. Imports and output directory

In [ ]:
import os, re, json, time, shutil, hashlib, platform, unicodedata, warnings
from datetime import datetime, timezone
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from datasets import load_dataset
from nltk.tokenize import sent_tokenize
from sklearn.metrics import silhouette_score
import umap

import opencc
import cyrtranslit
import pykakasi
from aksharamukha import transliterate as _aksharamukha
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator
from transformers import (
    AutoTokenizer, AutoModel, T5EncoderModel, AutoModelForMaskedLM
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## 3. Configuration and reproducibility logger

In [ ]:
ENCODER_MODELS = {
    'xlm-r': 'xlm-roberta-large',
    'byt5':  'google/byt5-large',
    'mt5':   'google/mt5-large',
}

# Production dataset sizes.
N_WORD_PAIRS  = 500   # unique word-level pairs per language
N_SENT_PAIRS  = 2000  # total sentence pairs per language
N_EVAL_PAIRS  = 500   # held-out pairs used for retrieval eval
                      # remaining N_SENT_PAIRS - N_EVAL_PAIRS used for training
                      # retrieval pool = all N_SENT_PAIRS script-B sentences

# Layer fractions probed (resolved to concrete indices per model).
LAYER_FRACS = [0.0, 0.25, 0.5, 0.75, 1.0]

# Derived: training pairs available for contrastive alignment
N_TRAIN_PAIRS = N_SENT_PAIRS - N_EVAL_PAIRS   # 1500

# Random baseline for retrieval (1 correct key in pool of N_SENT_PAIRS)
RETRIEVAL_RANDOM_BASELINE = 1.0 / N_SENT_PAIRS

RUN_TS  = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
RUN_DIR = Path('/kaggle/working') / f'run_{RUN_TS}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

for sub in ['corpus', 'embeddings', 'exp1', 'exp2', 'exp3', 'exp4',
            'exp5', 'exp6', 'figures']:
    (RUN_DIR / sub).mkdir(exist_ok=True)

print(f'Output directory      : {RUN_DIR}')
print(f'Sentences per language: {N_SENT_PAIRS}')
print(f'  training split      : {N_TRAIN_PAIRS}')
print(f'  eval split          : {N_EVAL_PAIRS}')
print(f'  retrieval pool size : {N_SENT_PAIRS} (all script-B sentences)')
print(f'  random baseline MRR : {RETRIEVAL_RANDOM_BASELINE:.6f}')

In [ ]:
# Reproducibility manifest — records every file saved, its SHA-256 hash,
# and the full environment, so any result can be traced back to its source.

import transformers as _tf, sklearn as _sk

MANIFEST = {
    'run_id'  : RUN_TS,
    'started' : datetime.now(timezone.utc).isoformat(),
    'config'  : {
        'N_WORD_PAIRS' : N_WORD_PAIRS,
        'N_SENT_PAIRS' : N_SENT_PAIRS,
        'LAYER_FRACS'  : LAYER_FRACS,
        'ENCODER_MODELS': ENCODER_MODELS,
        'DEVICE'       : DEVICE,
    },
    'environment': {
        'python'      : platform.python_version(),
        'torch'       : torch.__version__,
        'transformers': _tf.__version__,
        'sklearn'     : _sk.__version__,
        'cuda'        : torch.version.cuda,
        'gpu'         : torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu',
    },
    'files': [],
}


def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def save(rel_path, data, kind='json', description=''):
    """
    Write data to RUN_DIR/<rel_path> and record the event in the manifest.

    kind : 'json' | 'csv' | 'npz' | 'pdf' | 'copy'
        'npz'  — data must be dict[str, np.ndarray]
        'pdf'  — data must be a matplotlib Figure
        'copy' — data must be a path string; the file is copied verbatim
    """
    dest = RUN_DIR / rel_path
    dest.parent.mkdir(parents=True, exist_ok=True)

    if kind == 'json':
        with open(dest, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2, default=str)
    elif kind == 'csv':
        data.to_csv(dest, index=False)
    elif kind == 'npz':
        np.savez_compressed(dest, **data)
    elif kind == 'pdf':
        data.savefig(dest, dpi=150, bbox_inches='tight')
    elif kind == 'copy':
        shutil.copy2(data, dest)
    else:
        raise ValueError(f'Unknown kind: {kind}')

    MANIFEST['files'].append({
        'rel_path'   : rel_path,
        'kind'       : kind,
        'description': description,
        'sha256'     : _sha256(dest),
        'size_bytes' : dest.stat().st_size,
        'saved_at'   : datetime.now(timezone.utc).isoformat(),
    })
    _flush()
    return dest


def _flush():
    with open(RUN_DIR / 'manifest.json', 'w', encoding='utf-8') as f:
        json.dump(MANIFEST, f, ensure_ascii=False, indent=2, default=str)


def savefig(name, fig=None, description=''):
    if fig is None:
        fig = plt.gcf()
    return save(f'figures/{name}', fig, kind='pdf', description=description)


_flush()
print('Manifest initialised.')

In [ ]:
kaz_tr = Transliterator('kaz', Script.CYRILLIC, Script.LATIN)
ug_tr  = Transliterator('uig', Script.PERSO_ARABIC, Script.LATIN)
kaz_tr_reverse = Transliterator('kaz', Script.LATIN, Script.CYRILLIC)
ug_tr_reverse  = Transliterator('uig', Script.LATIN, Script.PERSO_ARABIC)
s2t    = opencc.OpenCC('s2t')
t2s    = opencc.OpenCC('t2s')
kks    = pykakasi.kakasi()

def kanji_to_hira(text):
    return ''.join(item['hira'] for item in kks.convert(text))

def contains_kanji(text):
    return any(unicodedata.name(c, '').startswith('CJK') for c in text)

def hi_to_ur(text):
    return _aksharamukha.process('Devanagari', 'Urdu', text)

print('Transliterators ready.')

In [ ]:
def fetch_wiki(lang, n, min_w=5, max_w=25, max_articles=5000):
    ds = load_dataset('wikimedia/wikipedia', f'20231101.{lang}',
                      split='train', streaming=True)
    sents, k = [], 0
    for article in ds:
        k += 1
        text = article.get('text', '')
        raw  = re.split(r'[。！？\n،؟]+', text) if lang in ('zh', 'ja', 'ug') \
               else sent_tokenize(text)
        for s in raw:
            s = s.strip()
            if min_w < len(s.split()) < max_w and len(s) > 10:
                sents.append(s)
        if len(sents) >= n or k >= max_articles:
            break
    return sents[:n]


def fetch_opus(lang, pair, n):
    ds = load_dataset('Helsinki-NLP/opus-100', pair,
                      split='train', streaming=True)
    sents = []
    for item in ds:
        s = item['translation'].get(lang, '').strip()
        if s and 3 < len(s.split()) < 40:
            sents.append(s)
        if len(sents) >= n:
            break
    return sents


def words_from_pairs(pairs, n, min_len=2):
    def dedup(it):
        seen = set()
        return [x for x in it if x not in seen and not seen.add(x)]
    wa = dedup(w for s, _ in pairs for w in s.split() if len(w) >= min_len)
    wb = dedup(w for _, s in pairs for w in s.split() if len(w) >= min_len)
    return wa[:n], wb[:n]


def persist(lang, pairs, words, scripts, source, tool, extra=None):
    """Save corpus files for one language and log them to the manifest."""
    wa, wb = words
    meta = {
        'language': lang, 'script_a': scripts[0], 'script_b': scripts[1],
        'n_pairs': len(pairs), 'n_words_a': len(wa), 'n_words_b': len(wb),
        'source': source, 'conversion_tool': tool,
        'built_at': datetime.now(timezone.utc).isoformat(),
    }
    if extra:
        meta.update(extra)
    save(f'corpus/{lang}_pairs.json',   [[a, b] for a, b in pairs],
         description=f'{lang} sentence pairs')
    save(f'corpus/{lang}_words_a.json', wa,
         description=f'{lang} script-A word list')
    save(f'corpus/{lang}_words_b.json', wb,
         description=f'{lang} script-B word list')
    save(f'corpus/{lang}_meta.json',    meta,
         description=f'{lang} corpus metadata')


print('Corpus utilities ready.')

In [ ]:
print('[1/6] Kazakh  — Cyrillic ↔ Latin  (low pretraining data)')
kaz_cyr   = fetch_wiki('kk', N_SENT_PAIRS)
KAZ_PAIRS = [(s, kaz_tr.transliterate(s)) for s in kaz_cyr]
KAZ_WORDS = words_from_pairs(KAZ_PAIRS, N_WORD_PAIRS)
persist('kazakh', KAZ_PAIRS, KAZ_WORDS, ('Cyrillic','Latin'),
        'kk Wikipedia 20231101', 'turkicnlp')
print(f'  {len(KAZ_PAIRS)} pairs | {len(KAZ_WORDS[0])} word types')

print('[2/6] Serbian — Cyrillic ↔ Latin  (high pretraining data)')
srb_cyr   = fetch_wiki('sr', N_SENT_PAIRS)
SRB_PAIRS = [(s, cyrtranslit.to_latin(s, 'sr')) for s in srb_cyr]
SRB_WORDS = words_from_pairs(SRB_PAIRS, N_WORD_PAIRS)
persist('serbian', SRB_PAIRS, SRB_WORDS, ('Cyrillic','Latin'),
        'sr Wikipedia 20231101', 'cyrtranslit')
print(f'  {len(SRB_PAIRS)} pairs | {len(SRB_WORDS[0])} word types')

print('[3/6] Chinese — Simplified ↔ Traditional')
zh_simp   = fetch_wiki('zh', N_SENT_PAIRS)
ZH_PAIRS  = [(s, s2t.convert(s)) for s in zh_simp]
ZH_WORDS  = words_from_pairs(ZH_PAIRS, N_WORD_PAIRS)
ZH_DIV    = [any(cs != t2s.convert(ct) for cs, ct in zip(a, b))
             for a, b in ZH_PAIRS]
persist('chinese', ZH_PAIRS, ZH_WORDS, ('Simplified','Traditional'),
        'zh Wikipedia 20231101', 'opencc s2t',
        extra={'n_divergent': sum(ZH_DIV)})
save('corpus/chinese_divergent_flags.json', ZH_DIV,
     description='Per-sentence lexical-divergence flag for Chinese')
print(f'  {len(ZH_PAIRS)} pairs | {sum(ZH_DIV)}/{len(ZH_PAIRS)} lexically divergent')

print('[4/6] Hindi–Urdu — Devanagari ↔ Nastaliq')
# Hindi from OPUS en-hi; Urdu script generated via aksharamukha.
# This is a script-level conversion; Hindustani vocabulary is identical on both sides.
hi_raw = fetch_opus('hi', 'en-hi', N_SENT_PAIRS * 3)
HI_PAIRS, n_fail = [], 0
for hi in hi_raw:
    try:
        ur = hi_to_ur(hi)
    except Exception:
        n_fail += 1
        continue
    if ur and ur != hi:
        HI_PAIRS.append((hi, ur))
    if len(HI_PAIRS) >= N_SENT_PAIRS:
        break
HI_WORDS = words_from_pairs(HI_PAIRS, N_WORD_PAIRS)
persist('hindi_urdu', HI_PAIRS, HI_WORDS, ('Devanagari','Nastaliq'),
        'OPUS-100 en-hi', 'aksharamukha',
        extra={'n_conversion_errors': n_fail})
print(f'  {len(HI_PAIRS)} pairs | {len(HI_WORDS[0])} word types'
      + (f' | {n_fail} conversion errors' if n_fail else ''))

print('[5/6] Uyghur  — Arabic-script ↔ ULY Latin')
ug_raw    = fetch_opus('ug', 'en-ug', N_SENT_PAIRS * 3)
ug_source = 'OPUS-100 en-ug'
if len(ug_raw) < 10:
    ug_raw    = fetch_wiki('ug', N_SENT_PAIRS * 3)
    ug_source = 'ug Wikipedia 20231101'
UG_PAIRS = []
for arab in ug_raw:
    try:
        lat = ug_tr.transliterate(arab)
    except Exception:
        continue
    if sum(1 for c in lat if ord(c) < 256) / max(len(lat), 1) > 0.6:
        UG_PAIRS.append((arab, lat))
    if len(UG_PAIRS) >= N_SENT_PAIRS:
        break
UG_WORDS = words_from_pairs(UG_PAIRS, N_WORD_PAIRS)
persist('uyghur', UG_PAIRS, UG_WORDS, ('Arabic-script','ULY Latin'),
        ug_source, 'turkicnlp uig')
print(f'  {len(UG_PAIRS)} pairs | {len(UG_WORDS[0])} word types')

print('[6/6] Japanese — Kanji/Mixed ↔ Hiragana')
# Oversample 4× so the Kanji filter reliably yields N_SENT_PAIRS.
ja_raw   = fetch_wiki('ja', N_SENT_PAIRS * 4)
JA_PAIRS = []
for s in ja_raw:
    if not contains_kanji(s):
        continue
    try:
        h = kanji_to_hira(s)
        if len(h) > 5:
            JA_PAIRS.append((s, h))
    except Exception:
        pass
    if len(JA_PAIRS) >= N_SENT_PAIRS:
        break
JA_WORDS = words_from_pairs(JA_PAIRS, N_WORD_PAIRS)
persist('japanese', JA_PAIRS, JA_WORDS, ('Kanji/Mixed','Hiragana'),
        'ja Wikipedia 20231101', 'pykakasi')
print(f'  {len(JA_PAIRS)} pairs | {len(JA_WORDS[0])} word types')

DATASETS = {
    'kazakh'    : {'pairs': KAZ_PAIRS, 'words': KAZ_WORDS,
                   'scripts': ('Cyrillic','Latin')},
    'serbian'   : {'pairs': SRB_PAIRS, 'words': SRB_WORDS,
                   'scripts': ('Cyrillic','Latin')},
    'chinese'   : {'pairs': ZH_PAIRS,  'words': ZH_WORDS,
                   'scripts': ('Simplified','Traditional'), 'divergent': ZH_DIV},
    'uyghur'    : {'pairs': UG_PAIRS,  'words': UG_WORDS,
                   'scripts': ('Arabic-script','ULY Latin')},
    'japanese'  : {'pairs': JA_PAIRS,  'words': JA_WORDS,
                   'scripts': ('Kanji/Mixed','Hiragana')},
    'hindi_urdu': {'pairs': HI_PAIRS,  'words': HI_WORDS,
                   'scripts': ('Devanagari','Nastaliq')},
}

save('corpus/dataset_index.json',
     {l: {'scripts': d['scripts'], 'n_pairs': len(d['pairs']),
           'n_words': len(d['words'][0])}
      for l, d in DATASETS.items()},
     description='Dataset summary')

print()
print(f'{"Language":<12}  {"Pairs":>6}  {"Words":>6}')
print('-' * 28)
for lang, d in DATASETS.items():
    print(f'{lang:<12}  {len(d["pairs"]):>6}  {len(d["words"][0]):>6}')

---
## 5. Embedding extraction utilities

In [ ]:
def _n_layers(model):
    if hasattr(model, 'config') and hasattr(model.config, 'num_hidden_layers'):
        return model.config.num_hidden_layers
    if hasattr(model, 'encoder') and hasattr(model.encoder, 'config'):
        return model.encoder.config.num_layers
    return 12


def layer_indices(model, fracs=None):
    """
    Convert LAYER_FRACS to concrete hidden-state indices.
    Index 0 is the raw embedding layer; transformer block outputs start at 1.
    """
    if fracs is None:
        fracs = LAYER_FRACS
    n = _n_layers(model)
    return [max(1, min(n, 1 + int(round(f * (n - 1))))) for f in fracs]


def embed(texts, tok, model, idxs, batch_size=16, pooling='mean'):
    """
    Mean-pool (or CLS/max-pool) hidden states at each requested layer.
    Returns dict[layer_idx -> np.ndarray of shape (N, H)].
    """
    device = next(model.parameters()).device
    model.eval()
    buckets = defaultdict(list)

    for i in range(0, len(texts), batch_size):
        batch = texts[i: i + batch_size]
        try:
            enc = tok(batch, return_tensors='pt', padding=True,
                      truncation=True, max_length=128).to(device)
            with torch.no_grad():
                out = model(**enc, output_hidden_states=True)
            hs   = out.hidden_states
            mask = enc['attention_mask'].unsqueeze(-1).float()
            for li in idxs:
                h = hs[li]
                if pooling == 'cls':
                    pooled = h[:, 0]
                elif pooling == 'max':
                    pooled = h.max(dim=1).values
                else:
                    pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
                pooled = F.normalize(pooled, dim=-1)
                buckets[li].append(pooled.cpu().float().numpy())
        except Exception as e:
            if not hasattr(embed, '_warned'):
                print(f'  embedding batch {i//batch_size} failed: {e}')
                embed._warned = True

    if not buckets:
        dim = getattr(model.config, 'hidden_size', 1024)
        return {li: np.zeros((len(texts), dim)) for li in idxs}
    return {li: np.concatenate(arrs) for li, arrs in buckets.items()}


def save_emb(emb_dict, mname, lang, tag):
    """Persist a layer-keyed embedding dict as a compressed .npz file."""
    arrays = {f'layer_{li}': arr for li, arr in emb_dict.items()}
    save(f'embeddings/{mname}/{lang}_{tag}.npz', arrays, kind='npz',
         description=f'{mname} embeddings — {lang} ({tag})')


def paired_cos(a, b):
    """Row-wise cosine similarity. Returns (N,) array."""
    a = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-9)
    b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-9)
    return (a * b).sum(axis=1)


def load_enc(name, path):
    tok = AutoTokenizer.from_pretrained(path)
    cls = T5EncoderModel if any(x in name for x in ('byt5','mt5')) else AutoModel
    mod = cls.from_pretrained(path).to(DEVICE)
    return tok, mod


def unload(model):
    del model
    torch.cuda.empty_cache()


def compute_mrr(q, k, correct_idx):
    """
    Mean Reciprocal Rank. q/k are (N, D) and (M, D) numpy arrays.
    Returns (mrr_float, per_query_list).
    """
    qn = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-9)
    kn = k / (np.linalg.norm(k, axis=1, keepdims=True) + 1e-9)
    S  = qn @ kn.T
    rr, per_query = 0.0, []
    for i, ci in enumerate(correct_idx):
        rank = int(np.where(np.argsort(-S[i]) == ci)[0][0]) + 1
        rr  += 1.0 / rank
        per_query.append({'query_idx': i, 'rank': rank, 'rr': 1.0 / rank})
    return rr / len(correct_idx), per_query


print('Embedding utilities ready.')

In [ ]:
layerwise = {}

for mname, mpath in ENCODER_MODELS.items():
    print(f'\n{mname}')
    tok, mod = load_enc(mname, mpath)
    idxs = layer_indices(mod)
    layerwise[mname] = {}

    for lang, d in DATASETS.items():
        wa, wb = list(d['words'][0]), list(d['words'][1])
        n = min(len(wa), len(wb))
        if n == 0:
            continue
        wa, wb = wa[:n], wb[:n]

        ea = embed(wa, tok, mod, idxs)
        eb = embed(wb, tok, mod, idxs)
        save_emb(ea, mname, lang, 'words_a')
        save_emb(eb, mname, lang, 'words_b')

        res = {}
        for li, frac in zip(idxs, LAYER_FRACS):
            sims = paired_cos(ea[li], eb[li])
            res[frac] = {'mean': float(np.nanmean(sims)),
                         'std':  float(np.nanstd(sims)),
                         'n':    n}
        layerwise[mname][lang] = res

        row = '  '.join(f'L{int(f*100):3d}%={v["mean"]:.3f}'
                        for f, v in res.items())
        print(f'  {lang:<12} {row}')

    save(f'exp1/{mname}_layerwise.json', layerwise[mname],
         description=f'Exp 1 layer-wise cosine similarity — {mname}')
    unload(mod)

save('exp1/all_layerwise.json', layerwise,
     description='Exp 1 layer-wise cosine similarity — all models')
print('\nExperiment 1 complete.')

In [ ]:
lang_colors = {
    'kazakh':'#1f77b4','serbian':'#ff7f0e','chinese':'#2ca02c',
    'uyghur':'#9467bd','japanese':'#8c564b','hindi_urdu':'#d62728',
}
lang_labels = {
    'kazakh':'Kazakh','serbian':'Serbian','chinese':'Chinese',
    'uyghur':'Uyghur','japanese':'Japanese','hindi_urdu':'Hindi–Urdu',
}

fig, axes = plt.subplots(1, len(ENCODER_MODELS),
                          figsize=(6*len(ENCODER_MODELS), 4.5), sharey=True)
for ax, (mname, lang_data) in zip(axes, layerwise.items()):
    for lang, res in lang_data.items():
        xs = list(res.keys())
        ys = [v['mean'] for v in res.values()]
        es = [v['std']  for v in res.values()]
        ax.plot(xs, ys, marker='o', label=lang_labels[lang],
                color=lang_colors[lang], linewidth=1.8)
        ax.fill_between(xs, [y-e for y,e in zip(ys,es)],
                            [y+e for y,e in zip(ys,es)],
                        alpha=0.12, color=lang_colors[lang])
    ax.set_title(mname, fontweight='bold')
    ax.set_xlabel('Layer depth')
    ax.set_xticks(LAYER_FRACS)
    ax.set_xticklabels(['0%','25%','50%','75%','100%'])
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Mean paired cosine similarity (±1 SD)')
fig.suptitle('Layer-wise script similarity by model and language',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
savefig('exp1_layerwise.pdf', fig, description='Exp 1 figure')
plt.show()

In [ ]:
cluster_res = {}

for mname, mpath in ENCODER_MODELS.items():
    print(f'\n{mname}')
    tok, mod = load_enc(mname, mpath)
    idxs = layer_indices(mod, fracs=[1.0])
    cluster_res[mname] = {}

    for lang, d in DATASETS.items():
        sa = [p[0] for p in d['pairs'][:N_SENT_PAIRS]]
        sb = [p[1] for p in d['pairs'][:N_SENT_PAIRS]]
        labels = [0]*len(sa) + [1]*len(sb)

        E_dict = embed(sa + sb, tok, mod, idxs)
        E      = E_dict[idxs[0]]
        sil    = silhouette_score(E, labels)

        save_emb({idxs[0]: E}, mname, lang, 'sents')
        cluster_res[mname][lang] = {
            'embeddings': E, 'labels': labels, 'silhouette': sil
        }
        print(f'  {lang:<12} silhouette = {sil:.3f}')

    save(f'exp2/{mname}_silhouette.json',
         {l: r['silhouette'] for l, r in cluster_res[mname].items()},
         description=f'Exp 2 silhouette scores — {mname}')
    unload(mod)

print('\nExperiment 2 complete.')

In [ ]:
reducer       = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
script_pal    = ['#4292c6','#ef6548']
n_m, n_l      = len(ENCODER_MODELS), len(DATASETS)
fig, axes     = plt.subplots(n_m, n_l, figsize=(4.5*n_l, 3.5*n_m))
umap_coords   = {}

for mi, (mname, ld) in enumerate(cluster_res.items()):
    umap_coords[mname] = {}
    for li, (lang, res) in enumerate(ld.items()):
        ax  = axes[mi][li]
        E2  = reducer.fit_transform(res['embeddings'])
        sl  = np.array(res['labels'])
        umap_coords[mname][lang] = E2.tolist()
        for sc, sc_name in enumerate(DATASETS[lang]['scripts']):
            m = sl == sc
            ax.scatter(E2[m,0], E2[m,1], c=script_pal[sc],
                       alpha=0.45, s=8, label=sc_name)
        ax.set_title(f'{mname} | {lang}\nsil={res["silhouette"]:.3f}', fontsize=8)
        ax.legend(fontsize=6, markerscale=2)
        ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('UMAP of sentence embeddings (coloured by script)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
savefig('exp2_umap.pdf', fig, description='Exp 2 UMAP figure')
plt.show()

save('exp2/umap_coordinates.json', umap_coords,
     description='Exp 2 UMAP 2D coordinates')

sil_rows = [
    {'Model': m, 'Language': l, 'Silhouette': round(r['silhouette'], 3)}
    for m, ld in cluster_res.items() for l, r in ld.items()
]
sil_df = pd.DataFrame(sil_rows)
save('exp2/silhouette_scores.csv', sil_df, kind='csv',
     description='Exp 2 silhouette scores — all models')

lang_order = list(DATASETS.keys())
sil_pivot  = sil_df.pivot(index='Model', columns='Language',
                           values='Silhouette')[lang_order]
fig2, ax2  = plt.subplots(figsize=(10, 3))
sns.heatmap(sil_pivot, ax=ax2, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=-0.1, vmax=0.6, linewidths=0.5,
            cbar_kws={'label': 'Script silhouette'})
ax2.set_title('Script cluster silhouette by model and language\n'
              '(higher = stronger script-based separation)', fontweight='bold')
ax2.set_xlabel('')
plt.tight_layout()
savefig('exp2_silhouette_heatmap.pdf', fig2,
        description='Exp 2 silhouette heatmap')
plt.show()

In [ ]:
print('Loading XLM-R for MLM probing...')
tok_mlm = AutoTokenizer.from_pretrained(ENCODER_MODELS['xlm-r'])
mod_mlm = AutoModelForMaskedLM.from_pretrained(ENCODER_MODELS['xlm-r']).to(DEVICE)
mod_mlm.eval()

mlm_results = {}

for lang, d in DATASETS.items():
    wa, wb       = list(d['words'][0])[:500], list(d['words'][1])[:500]
    per_pair_log = []
    prefer_a, total = 0, 0

    for word_a, word_b in zip(wa, wb):
        ta = tok_mlm.tokenize(word_a)
        tb = tok_mlm.tokenize(word_b)
        if not ta or not tb:
            continue

        text = tok_mlm.convert_tokens_to_string([tok_mlm.mask_token] + ta[1:])
        enc  = tok_mlm(text, return_tensors='pt',
                       truncation=True, max_length=64).to(DEVICE)
        mpos = (enc['input_ids'] == tok_mlm.mask_token_id).nonzero(as_tuple=True)
        if len(mpos[0]) == 0:
            continue

        id_a = tok_mlm.convert_tokens_to_ids(ta[0])
        id_b = tok_mlm.convert_tokens_to_ids(tb[0])
        if id_a == tok_mlm.unk_token_id or id_b == tok_mlm.unk_token_id:
            continue

        with torch.no_grad():
            logits = mod_mlm(**enc).logits[0, mpos[1][0].item()]

        la, lb    = float(logits[id_a].cpu()), float(logits[id_b].cpu())
        pa        = la > lb
        prefer_a += int(pa)
        total    += 1
        per_pair_log.append({
            'word_a': word_a, 'word_b': word_b,
            'token_a': ta[0],  'token_b': tb[0],
            'logit_a': la,     'logit_b': lb,
            'prefers_a': pa,
        })

    pref = prefer_a / total if total > 0 else float('nan')
    mlm_results[lang] = {'prefer_a': pref, 'n': total}
    save(f'exp3/xlm-r_{lang}_per_pair.json', per_pair_log,
         description=f'Exp 3 MLM per-pair logits — {lang}')
    print(f'  {lang:<12} prefers {d["scripts"][0]} = {pref:.1%}  (n={total})')

save('exp3/xlm-r_aggregate.json', mlm_results,
     description='Exp 3 MLM aggregate script preference')
unload(mod_mlm)
print('\nExperiment 3 complete.')

In [ ]:
retrieval_results = {}

for mname, mpath in ENCODER_MODELS.items():
    print(f'\n{mname}')
    tok, mod = load_enc(mname, mpath)
    idxs     = layer_indices(mod, fracs=[1.0])
    retrieval_results[mname] = {}

    for lang, d in DATASETS.items():
        # Eval queries: the last N_EVAL_PAIRS sentence pairs (held out from training)
        eval_pairs  = d['pairs'][-N_EVAL_PAIRS:]
        sents_a     = [p[0] for p in eval_pairs]
        sents_b_gt  = [p[1] for p in eval_pairs]

        # Retrieval pool: ALL script-B sentences in the language (no cycling,
        # no modular arithmetic). Correct keys are the last N_EVAL_PAIRS entries.
        all_b_sents = [p[1] for p in d['pairs']]   # all 2000 script-B sentences
        key_pool    = all_b_sents
        # Correct key for eval_pairs[i] is d['pairs'][-(N_EVAL_PAIRS - i)]
        # i.e., the last N_EVAL_PAIRS entries in key_pool
        correct_idx = list(range(N_SENT_PAIRS - N_EVAL_PAIRS, N_SENT_PAIRS))

        ea = embed(sents_a,  tok, mod, idxs)
        ek = embed(key_pool, tok, mod, idxs)
        save_emb(ea, mname, lang, 'retrieval_q')
        save_emb(ek, mname, lang, 'retrieval_k')

        save(f'exp4/{mname}_{lang}_pool.json',
             {'n_queries':     N_EVAL_PAIRS,
              'pool_size':     len(key_pool),
              'correct_indices': correct_idx,
              'random_baseline': RETRIEVAL_RANDOM_BASELINE},
             description=f'Exp 4 retrieval pool metadata — {mname} × {lang}')

        mrr, pq = compute_mrr(ea[idxs[0]], ek[idxs[0]], correct_idx)
        retrieval_results[mname][lang] = mrr
        save(f'exp4/{mname}_{lang}_per_query.json', pq,
             description=f'Exp 4 per-query ranks — {mname} × {lang}')
        print(f'  {lang:<12} MRR = {mrr:.4f}  '
              f'(pool={len(key_pool)}, random={RETRIEVAL_RANDOM_BASELINE:.6f})')

    save(f'exp4/{mname}_mrr.json', retrieval_results[mname],
         description=f'Exp 4 MRR — {mname}')
    unload(mod)

save('exp4/all_mrr.json', retrieval_results,
     description='Exp 4 MRR — all models')
print('\nExperiment 4 complete.')

In [ ]:
def fl_sim(mname, lang):
    vals = layerwise.get(mname, {}).get(lang, {})
    return vals[max(vals.keys())]['mean'] if vals else float('nan')


decomp_rows = []
for lang in DATASETS:
    xlmr = fl_sim('xlm-r', lang)
    byt5 = fl_sim('byt5',  lang)
    mt5  = fl_sim('mt5',   lang)
    decomp_rows.append({
        'Language'       : lang,
        'XLM-R sim'      : round(xlmr, 3),
        'byT5 sim'       : round(byt5, 3),
        'mT5 sim'        : round(mt5,  3),
        'Tok. effect'    : round(xlmr - byt5, 3),
        'Residual (byT5)': round(byt5, 3),
        'MRR (XLM-R)'    : round(retrieval_results.get('xlm-r',{}).get(lang, float('nan')), 3),
        'MRR (byT5)'     : round(retrieval_results.get('byt5', {}).get(lang, float('nan')), 3),
    })

decomp_df       = pd.DataFrame(decomp_rows)
data_vol_effect = fl_sim('xlm-r','serbian') - fl_sim('xlm-r','kazakh')

print('Causal decomposition')
print(decomp_df.to_string(index=False))
print(f'\nData volume effect (Serbian − Kazakh, XLM-R): {data_vol_effect:+.4f}')

save('exp5/decomposition.csv', decomp_df, kind='csv',
     description='Exp 5 causal decomposition table')
save('exp5/decomposition.json',
     {'table': decomp_rows, 'data_volume_effect': data_vol_effect},
     description='Exp 5 causal decomposition (JSON)')

# Heatmaps
lang_order = list(DATASETS.keys())
mrr_rows   = [{'Model': m, 'Language': l, 'MRR': round(v, 3)}
              for m, ld in retrieval_results.items() for l, v in ld.items()]
sim_rows   = [{'Model': m, 'Language': l,
               'Final-layer sim': round(fl_sim(m, l), 3)}
              for m in ENCODER_MODELS for l in DATASETS]

sim_pivot = pd.DataFrame(sim_rows).pivot(
    index='Model', columns='Language', values='Final-layer sim')[lang_order]
mrr_pivot = pd.DataFrame(mrr_rows).pivot(
    index='Model', columns='Language', values='MRR').reindex(columns=lang_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, pivot, title, label in [
    (axes[0], sim_pivot, 'Final-layer cosine similarity', 'Cosine sim'),
    (axes[1], mrr_pivot, 'Cross-script retrieval MRR',   'MRR'),
]:
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label': label})
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
fig.suptitle('Causal decomposition: similarity and retrieval by model and language',
             fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('exp5_decomposition_heatmaps.pdf', fig,
        description='Exp 5 causal decomposition heatmaps')
plt.show()
print('\nExperiment 5 complete.')

In [ ]:
NORMALIZERS = {
    # Kazakh: corpus built Cyrillic→Latin, so normalise Latin→Cyrillic
    'kazakh'    : kaz_tr_reverse.transliterate,
    # Serbian: corpus built Cyrillic→Latin, so normalise Latin→Cyrillic
    'serbian'   : lambda s: cyrtranslit.to_cyrillic(s, 'sr'),
    # Chinese: corpus built Simplified→Traditional, so normalise Trad→Simp
    'chinese'   : t2s.convert,   # unchanged — already correct direction
    # Uyghur: corpus built Arabic→Latin, so normalise Latin→Arabic
    'uyghur'    : ug_tr_reverse.transliterate,
    # Japanese: Kanji→Hiragana is not invertible (many Kanji share readings)
    # so we normalise in the forward direction: convert Kanji queries to
    # Hiragana so both sides are Hiragana. Apply to queries, not keys.
    'japanese'  : kanji_to_hira,
    # Hindi-Urdu: corpus built Devanagari→Nastaliq via aksharamukha.
    # Normalise Nastaliq→Devanagari using the reverse direction.
    'hindi_urdu': lambda s: _aksharamukha.process('Urdu', 'Devanagari', s),
}

def whiten_fit(X_train):
    """
    Fit whitening parameters on training embeddings.
    Returns (mu, W) where W is the whitening matrix.
    Fitting on held-out eval data would leak information into the
    whitening transform; we fit on training embeddings only.
    """
    mu  = X_train.mean(axis=0, keepdims=True)
    Xc  = X_train - mu
    U, S, _ = np.linalg.svd(np.cov(Xc, rowvar=False))
    W   = U @ np.diag(1.0 / np.sqrt(S + 1e-5))
    return mu, W


def whiten_apply(X, mu, W):
    return (X - mu) @ W


def csls_sim(x, y, k=10):
    sim  = x @ y.T
    r_x  = np.mean(np.sort(sim, axis=1)[:, -k:], axis=1)
    r_y  = np.mean(np.sort(sim, axis=0)[-k:, :], axis=0)
    return 2 * sim - r_x[:, None] - r_y[None, :]


def mrr_csls(q, k, correct_idx):
    qn, kn = (v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)
              for v in (q, k))
    S  = csls_sim(qn, kn)
    rr = [1.0 / (np.where(np.argsort(-S[i]) == ci)[0][0] + 1)
          for i, ci in enumerate(correct_idx)]
    return float(np.mean(rr))


class PairDataset(Dataset):
    def __init__(self, a, b):
        self.a = torch.tensor(a, dtype=torch.float32)
        self.b = torch.tensor(b, dtype=torch.float32)
    def __len__(self):        return len(self.a)
    def __getitem__(self, i): return self.a[i], self.b[i]


class ProjectionHead(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 512), nn.ReLU(), nn.Linear(512, 256)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


def nt_xent(z1, z2, tau=0.05):
    logits = (z1 @ z2.T) / tau
    labels = torch.arange(len(z1), device=z1.device)
    return (F.cross_entropy(logits, labels) +
            F.cross_entropy(logits.T, labels)) / 2


def encode_all_layers(texts, tok, model, batch_size=16):
    """
    Single forward pass returning hidden states at every layer.
    Returns dict {layer_idx: (N, D) array}.
    More efficient than re-encoding per layer in the layer search.
    """
    n_total  = _n_layers(model) + 1   # embedding layer + transformer blocks
    all_idxs = list(range(n_total))
    buckets  = defaultdict(list)
    model.eval()
    device = next(model.parameters()).device

    for i in range(0, len(texts), batch_size):
        batch = texts[i: i + batch_size]
        enc   = tok(batch, return_tensors='pt', padding=True,
                    truncation=True, max_length=128).to(device)
        with torch.no_grad():
            out  = model(**enc, output_hidden_states=True)
        hs   = out.hidden_states
        mask = enc['attention_mask'].unsqueeze(-1).float()
        for li in all_idxs:
            pooled = (hs[li] * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            pooled = F.normalize(pooled, dim=-1)
            buckets[li].append(pooled.cpu().float().numpy())

    return {li: np.concatenate(arrs) for li, arrs in buckets.items()}


print('Intervention utilities ready.')

In [ ]:
print('Loading XLM-R for intervention study...')
tok_x, mod_x = load_enc('xlm-r', ENCODER_MODELS['xlm-r'])
mod_x.eval()
n_layers    = _n_layers(mod_x)
final_layer = n_layers

int_results = {}

for lang, d in DATASETS.items():
    print(f'\n{"="*60}\n{lang.upper()}\n{"="*60}')

    # Split: first N_TRAIN_PAIRS for training, last N_EVAL_PAIRS for eval.
    # This matches Experiment 4 exactly — eval queries are identical.
    train_pairs = d['pairs'][:N_TRAIN_PAIRS]
    eval_pairs  = d['pairs'][-N_EVAL_PAIRS:]

    sents_a     = [p[0] for p in eval_pairs]
    sents_b     = [p[1] for p in eval_pairs]

    # Retrieval pool: all script-B sentences (same as Experiment 4)
    all_b_sents = [p[1] for p in d['pairs']]
    key_pool_b  = all_b_sents
    correct_idx = list(range(N_SENT_PAIRS - N_EVAL_PAIRS, N_SENT_PAIRS))

    # --- Baseline ---
    print('Caching baseline embeddings...')
    ea = embed(sents_a,   tok_x, mod_x, [final_layer])[final_layer]
    eb = embed(key_pool_b, tok_x, mod_x, [final_layer])[final_layer]
    mrr_base, pq_base = compute_mrr(ea, eb, correct_idx)
    print(f'  Baseline MRR: {mrr_base:.3f}')

    save_emb({final_layer: ea}, 'xlm-r', lang, 'int_query')
    save_emb({final_layer: eb}, 'xlm-r', lang, 'int_key_baseline')

    # --- Script normalisation ---
    print('Running normalisation intervention...')
    try:
        norm          = NORMALIZERS[lang]
        sents_b_n     = [norm(s) for s in sents_b]
        key_pool_n    = [norm(s) for s in key_pool_b]
        eb_n = embed(key_pool_n, tok_x, mod_x, [final_layer])[final_layer]
        mrr_norm, pq_norm = compute_mrr(ea, eb_n, correct_idx)
        save_emb({final_layer: eb_n}, 'xlm-r', lang, 'int_key_norm')
        save(f'exp6/{lang}_norm_texts.json',
             {'eval_b_original': sents_b, 'eval_b_normalised': sents_b_n},
             description=f'Exp 6 normalised eval texts — {lang}')
    except Exception as e:
        print(f'  normalisation failed: {e}')
        mrr_norm, pq_norm = float('nan'), []

    print(f'  Normalised MRR: {mrr_norm:.3f}')

    # --- Whitening ---
    # IMPORTANT: whitening parameters are estimated on TRAINING embeddings only,
    # then applied to eval embeddings. Estimating on eval data would leak
    # information about the test distribution into the transform.
    print('Running whitening...')
    train_sa_sents = [p[0] for p in train_pairs]
    train_sb_sents = [p[1] for p in train_pairs]
    train_ea = embed(train_sa_sents, tok_x, mod_x, [final_layer])[final_layer]
    train_eb = embed(train_sb_sents, tok_x, mod_x, [final_layer])[final_layer]
    train_combined = np.concatenate([train_ea, train_eb])

    mu_w, W_w = whiten_fit(train_combined)

    ea_w = whiten_apply(ea, mu_w, W_w)
    eb_w = whiten_apply(eb, mu_w, W_w)
    mrr_white, _ = compute_mrr(ea_w, eb_w, correct_idx)
    print(f'  Whitened MRR: {mrr_white:.3f}')

    # --- CSLS ---
    print('Running CSLS retrieval...')
    mrr_csls_val = mrr_csls(ea, eb, correct_idx)
    print(f'  CSLS MRR: {mrr_csls_val:.3f}')

    # --- Layer search (single-pass — all layers cached in one forward pass) ---
    print('Running layer search...')
    q_by_layer = encode_all_layers(sents_a,   tok_x, mod_x)
    k_by_layer = encode_all_layers(key_pool_b, tok_x, mod_x)

    layer_mrrs = []
    for li in range(n_layers + 1):
        m, _ = compute_mrr(q_by_layer[li], k_by_layer[li], correct_idx)
        layer_mrrs.append(float(m))

    best_layer     = int(np.argmax(layer_mrrs))
    best_layer_mrr = float(np.max(layer_mrrs))
    print(f'  Best layer: {best_layer} (MRR={best_layer_mrr:.3f})')

    # --- Contrastive alignment ---
    # Train ONLY on the training split (first N_TRAIN_PAIRS pairs).
    # The eval pairs are never seen during training, so this is a fair
    # measure of out-of-sample generalisation.
    print('Running contrastive alignment...')
    emb_ta = train_ea   # already computed above for whitening
    emb_tb = train_eb

    loader = DataLoader(PairDataset(emb_ta, emb_tb),
                        batch_size=64, shuffle=True)
    proj   = ProjectionHead(emb_ta.shape[1]).to(DEVICE)
    optim  = torch.optim.AdamW(proj.parameters(), lr=1e-4)

    proj.train()
    for epoch in range(10):
        total = 0.0
        for xa, xb in loader:
            xa, xb = xa.to(DEVICE), xb.to(DEVICE)
            loss   = nt_xent(proj(xa), proj(xb))
            optim.zero_grad(); loss.backward(); optim.step()
            total += loss.item()
        print(f'  epoch {epoch+1}/10  loss={total/len(loader):.4f}')

    proj.eval()
    with torch.no_grad():
        ea_c = proj(torch.tensor(ea, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        eb_c = proj(torch.tensor(eb, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    mrr_contr, _ = compute_mrr(ea_c, eb_c, correct_idx)
    print(f'  Contrastive MRR: {mrr_contr:.3f}')

    # --- Token overlap ---
    overlaps = []
    for wa, wb in zip(list(d['words'][0])[:100], list(d['words'][1])[:100]):
        ta = set(tok_x.tokenize(wa))
        tb = set(tok_x.tokenize(wb))
        if ta | tb:
            overlaps.append(len(ta & tb) / len(ta | tb))
    tok_overlap = float(np.mean(overlaps)) if overlaps else float('nan')

    save(f'exp6/{lang}_per_query.json',
         {'baseline': pq_base, 'normalised': pq_norm},
         description=f'Exp 6 per-query ranks — {lang}')
    save(f'exp6/{lang}_layer_mrrs.json', layer_mrrs,
         description=f'Exp 6 layer-wise MRR — {lang}')

    int_results[lang] = {
        'baseline_mrr'   : mrr_base,
        'normalised_mrr' : mrr_norm,
        'whitened_mrr'   : mrr_white,
        'csls_mrr'       : mrr_csls_val,
        'contrastive_mrr': mrr_contr,
        'best_layer'     : best_layer,
        'best_layer_mrr' : best_layer_mrr,
        'layer_mrrs'     : layer_mrrs,
        'token_overlap'  : tok_overlap,
    }
    save(f'exp6/{lang}_results.json', int_results[lang],
         description=f'Exp 6 full intervention results — {lang}')

    print(f'\n  Summary — {lang}')
    for k, v in [('Baseline',    mrr_base),    ('Normalised', mrr_norm),
                 ('Whitened',    mrr_white),   ('CSLS',       mrr_csls_val),
                 ('Contrastive', mrr_contr),   ('Best layer', best_layer_mrr)]:
        print(f'    {k:<14}: {v:.3f}')

save('exp6/all_results.json', int_results,
     description='Exp 6 all intervention results')
unload(mod_x)
print('\nExperiment 6 complete.')

In [ ]:
print('Loading XLM-R for intervention study...')
tok_x, mod_x = load_enc('xlm-r', ENCODER_MODELS['xlm-r'])
mod_x.eval()
n_layers    = _n_layers(mod_x)
final_layer = n_layers

int_results = {}

for lang, d in DATASETS.items():
    print(f'\n{"="*60}\n{lang.upper()}\n{"="*60}')

    # Split: first N_TRAIN_PAIRS for training, last N_EVAL_PAIRS for eval.
    # This matches Experiment 4 exactly — eval queries are identical.
    train_pairs = d['pairs'][:N_TRAIN_PAIRS]
    eval_pairs  = d['pairs'][-N_EVAL_PAIRS:]

    sents_a     = [p[0] for p in eval_pairs]
    sents_b     = [p[1] for p in eval_pairs]

    # Retrieval pool: all script-B sentences (same as Experiment 4)
    all_b_sents = [p[1] for p in d['pairs']]
    key_pool_b  = all_b_sents
    correct_idx = list(range(N_SENT_PAIRS - N_EVAL_PAIRS, N_SENT_PAIRS))

    # --- Baseline ---
    print('Caching baseline embeddings...')
    ea = embed(sents_a,   tok_x, mod_x, [final_layer])[final_layer]
    eb = embed(key_pool_b, tok_x, mod_x, [final_layer])[final_layer]
    mrr_base, pq_base = compute_mrr(ea, eb, correct_idx)
    print(f'  Baseline MRR: {mrr_base:.3f}')

    save_emb({final_layer: ea}, 'xlm-r', lang, 'int_query')
    save_emb({final_layer: eb}, 'xlm-r', lang, 'int_key_baseline')

    # --- Script normalisation ---
    print('Running normalisation intervention...')
    try:
        norm          = NORMALIZERS[lang]
        sents_b_n     = [norm(s) for s in sents_b]
        key_pool_n    = [norm(s) for s in key_pool_b]
        eb_n = embed(key_pool_n, tok_x, mod_x, [final_layer])[final_layer]
        mrr_norm, pq_norm = compute_mrr(ea, eb_n, correct_idx)
        save_emb({final_layer: eb_n}, 'xlm-r', lang, 'int_key_norm')
        save(f'exp6/{lang}_norm_texts.json',
             {'eval_b_original': sents_b, 'eval_b_normalised': sents_b_n},
             description=f'Exp 6 normalised eval texts — {lang}')
    except Exception as e:
        print(f'  normalisation failed: {e}')
        mrr_norm, pq_norm = float('nan'), []

    print(f'  Normalised MRR: {mrr_norm:.3f}')

    # --- Whitening ---
    # IMPORTANT: whitening parameters are estimated on TRAINING embeddings only,
    # then applied to eval embeddings. Estimating on eval data would leak
    # information about the test distribution into the transform.
    print('Running whitening...')
    train_sa_sents = [p[0] for p in train_pairs]
    train_sb_sents = [p[1] for p in train_pairs]
    train_ea = embed(train_sa_sents, tok_x, mod_x, [final_layer])[final_layer]
    train_eb = embed(train_sb_sents, tok_x, mod_x, [final_layer])[final_layer]
    train_combined = np.concatenate([train_ea, train_eb])

    mu_w, W_w = whiten_fit(train_combined)

    ea_w = whiten_apply(ea, mu_w, W_w)
    eb_w = whiten_apply(eb, mu_w, W_w)
    mrr_white, _ = compute_mrr(ea_w, eb_w, correct_idx)
    print(f'  Whitened MRR: {mrr_white:.3f}')

    # --- CSLS ---
    print('Running CSLS retrieval...')
    mrr_csls_val = mrr_csls(ea, eb, correct_idx)
    print(f'  CSLS MRR: {mrr_csls_val:.3f}')

    # --- Layer search (single-pass — all layers cached in one forward pass) ---
    print('Running layer search...')
    q_by_layer = encode_all_layers(sents_a,   tok_x, mod_x)
    k_by_layer = encode_all_layers(key_pool_b, tok_x, mod_x)

    layer_mrrs = []
    for li in range(n_layers + 1):
        m, _ = compute_mrr(q_by_layer[li], k_by_layer[li], correct_idx)
        layer_mrrs.append(float(m))

    best_layer     = int(np.argmax(layer_mrrs))
    best_layer_mrr = float(np.max(layer_mrrs))
    print(f'  Best layer: {best_layer} (MRR={best_layer_mrr:.3f})')

    # --- Contrastive alignment ---
    # Train ONLY on the training split (first N_TRAIN_PAIRS pairs).
    # The eval pairs are never seen during training, so this is a fair
    # measure of out-of-sample generalisation.
    print('Running contrastive alignment...')
    emb_ta = train_ea   # already computed above for whitening
    emb_tb = train_eb

    loader = DataLoader(PairDataset(emb_ta, emb_tb),
                        batch_size=64, shuffle=True)
    proj   = ProjectionHead(emb_ta.shape[1]).to(DEVICE)
    optim  = torch.optim.AdamW(proj.parameters(), lr=1e-4)

    proj.train()
    for epoch in range(10):
        total = 0.0
        for xa, xb in loader:
            xa, xb = xa.to(DEVICE), xb.to(DEVICE)
            loss   = nt_xent(proj(xa), proj(xb))
            optim.zero_grad(); loss.backward(); optim.step()
            total += loss.item()
        print(f'  epoch {epoch+1}/10  loss={total/len(loader):.4f}')

    proj.eval()
    with torch.no_grad():
        ea_c = proj(torch.tensor(ea, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        eb_c = proj(torch.tensor(eb, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    mrr_contr, _ = compute_mrr(ea_c, eb_c, correct_idx)
    print(f'  Contrastive MRR: {mrr_contr:.3f}')

    # --- Token overlap ---
    overlaps = []
    for wa, wb in zip(list(d['words'][0])[:100], list(d['words'][1])[:100]):
        ta = set(tok_x.tokenize(wa))
        tb = set(tok_x.tokenize(wb))
        if ta | tb:
            overlaps.append(len(ta & tb) / len(ta | tb))
    tok_overlap = float(np.mean(overlaps)) if overlaps else float('nan')

    save(f'exp6/{lang}_per_query.json',
         {'baseline': pq_base, 'normalised': pq_norm},
         description=f'Exp 6 per-query ranks — {lang}')
    save(f'exp6/{lang}_layer_mrrs.json', layer_mrrs,
         description=f'Exp 6 layer-wise MRR — {lang}')

    int_results[lang] = {
        'baseline_mrr'   : mrr_base,
        'normalised_mrr' : mrr_norm,
        'whitened_mrr'   : mrr_white,
        'csls_mrr'       : mrr_csls_val,
        'contrastive_mrr': mrr_contr,
        'best_layer'     : best_layer,
        'best_layer_mrr' : best_layer_mrr,
        'layer_mrrs'     : layer_mrrs,
        'token_overlap'  : tok_overlap,
    }
    save(f'exp6/{lang}_results.json', int_results[lang],
         description=f'Exp 6 full intervention results — {lang}')

    print(f'\n  Summary — {lang}')
    for k, v in [('Baseline',    mrr_base),    ('Normalised', mrr_norm),
                 ('Whitened',    mrr_white),   ('CSLS',       mrr_csls_val),
                 ('Contrastive', mrr_contr),   ('Best layer', best_layer_mrr)]:
        print(f'    {k:<14}: {v:.3f}')

save('exp6/all_results.json', int_results,
     description='Exp 6 all intervention results')
unload(mod_x)
print('\nExperiment 6 complete.')

---
## 12. Full results summary

In [ ]:
sep = '=' * 68
print(sep, 'RESULTS SUMMARY', sep, sep=('\n'))

print('\n--- Experiment 1: Final-layer cosine similarity ---')
e1 = [{'Model': m, 'Language': l, 'Final-layer sim': round(fl_sim(m,l),3)}
      for m in ENCODER_MODELS for l in DATASETS]
print(pd.DataFrame(e1).to_string(index=False))

print('\n--- Experiment 2: Script cluster silhouette ---')
print(sil_df.to_string(index=False))

print('\n--- Experiment 3: MLM script preference (XLM-R) ---')
for lang, res in mlm_results.items():
    print(f'  {lang:<12} prefers {DATASETS[lang]["scripts"][0]} = '
          f'{res["prefer_a"]:.1%}  (n={res["n"]})')

print('\n--- Experiment 4: Cross-script retrieval MRR ---')
print(pd.DataFrame(mrr_rows).to_string(index=False))

print('\n--- Experiment 5: Causal decomposition ---')
print(decomp_df.to_string(index=False))
print(f'  Data volume effect (Serbian − Kazakh, XLM-R): {data_vol_effect:+.4f}')

print('\n--- Experiment 6: Intervention study ---')
e6_rows = [
    {'Language'    : l,
     'Baseline'    : round(v['baseline_mrr'],    3),
     'Norm.'       : round(v['normalised_mrr'],  3),
     'Whitened'    : round(v['whitened_mrr'],    3),
     'CSLS'        : round(v['csls_mrr'],        3),
     'Contrastive' : round(v['contrastive_mrr'], 3),
     'Best layer'  : v['best_layer'],
     'Best MRR'    : round(v['best_layer_mrr'],  3),
     'Tok. overlap': round(v['token_overlap'],   3)}
    for l, v in int_results.items()
]
print(pd.DataFrame(e6_rows).to_string(index=False))

# Save full summary
full_summary = {
    'exp1_layerwise'   : e1,
    'exp2_silhouette'  : sil_df.to_dict(orient='records'),
    'exp3_mlm'         : mlm_results,
    'exp4_mrr'         : mrr_rows,
    'exp5_decomposition': decomp_rows,
    'exp5_data_vol_effect': data_vol_effect,
    'exp6_interventions': e6_rows,
}
save('full_summary.json', full_summary,
     description='Complete results summary — all experiments')

MANIFEST['completed'] = datetime.now(timezone.utc).isoformat()
_flush()

print(f'\n{sep}')
print(f'All outputs in : {RUN_DIR}')
print(f'Manifest       : {RUN_DIR}/manifest.json')
print(f'Files saved    : {len(MANIFEST["files"])}')
print(sep)